In [1]:
import numpy as np
import torch
from torch import nn
import tqdm

import torchvision
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split
import torch.nn.init as init

from sklearn.metrics import accuracy_score

In [2]:
# Use the following code to load and normalize the dataset for training and testing
# It will downlad the dataset into data subfolder (change to your data folder name)
train_dataset = torchvision.datasets.MNIST('/Users/jadeg/Documents/UW/winter/AMATH 582/HW4/data/', train=True, download=True,
                             transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),
                               torchvision.transforms.Normalize(
                                 (0.1307,), (0.3081,))
                             ]))

test_dataset = torchvision.datasets.MNIST('/Users/jadeg/Documents/UW/winter/AMATH 582/HW4/data/', train=False, download=True,
                             transform=torchvision.transforms.Compose([
                               torchvision.transforms.ToTensor(),
                               torchvision.transforms.Normalize(
                                 (0.1307,), (0.3081,))
                             ]))



In [3]:
# Use the following code to create a validation set of 10%
train_indices, val_indices, _, _ = train_test_split(
    range(len(train_dataset)),
    train_dataset.targets,
    stratify=train_dataset.targets,
    test_size=0.1,
)

# Generate training and validation subsets based on indices
train_split = Subset(train_dataset, train_indices)
val_split = Subset(train_dataset, val_indices)


In [4]:
#Define your (As Cool As It Gets) Fully Connected Neural Network 
class ACAIGFCN(nn.Module):
    #Initialize model layers, add additional arguments to adjust
    def __init__(self, input_dim, hidden_lays, output_dim, dropout_prob, init, batchnorm): 
        super(ACAIGFCN, self).__init__()
        #Define the network layer(s) and activation function(s)
        # create empty layers list to add layers to
        lays = []
        # set initial prev_dim as the input_dim
        prev_dim = input_dim

        # parse through each individual hidden layer, y=xAT+b
        for hidden_dim in hidden_lays:
            # apply linear transformation of input, append to layers
            lays.append(nn.Linear(prev_dim, hidden_dim))
            # apply batch norm
            if batchnorm=='yes':
                lays.append(nn.BatchNorm1d(hidden_dim))  
            # Applies theo rectified linear unit function element-wise, append to layers (introduces nonlinearlity)
            lays.append(nn.ReLU())
            # apply dropout
            lays.append(nn.Dropout(p=dropout_prob))
            # reset prev_dim as the current, hidden_dim
            prev_dim = hidden_dim
        
        # do one more linear transformation, getting to output_dim
        # no need for relu activation b/c cross entropy loss expects raw class scores
        lays.append(nn.Linear(prev_dim, output_dim))

        # sequential container - models are added to it as they are passed in constructor
        self.model = nn.Sequential(*lays) # hid,  hid2 ..

        if init=='RN' or init=='XN' or init=='KU':
            # apply initiation
            self.init_weights()
 
    def forward(self, input):
        #Define how your model propagates the input through the network
        return self.model(input)
    
    def init_weights(self):
        for layer in self.model:
            # Only initialize Linear layers
            if isinstance(layer, nn.Linear):
                if init=='RN':  
                    # Normal distribution
                    init.normal_(layer.weight, mean=0.0, std=0.01)
                elif init=='XN':
                    # Normal distribution
                    init.xavier_normal_(layer.weight) 
                elif init=='KU':
                    # Normal distribution
                    init.kaiming_uniform_(layer.weight, nonlinearity='relu')
                # Initialize bias to zero
                init.zeros_(layer.bias)  
 
    

In [5]:
# create a fxn to easily adjust parameters
def train_and_eval(hidden_lays, learning_rate, epochs, train_batch_size, opt, dropout_prob, init, batchnorm):
    # Initialize neural network model with input, output and hidden layer dimensions
    model = ACAIGFCN(input_dim = 784, hidden_lays= hidden_lays, output_dim = 10, dropout_prob=dropout_prob, init=init, batchnorm=batchnorm) 

    # Define dataloader objects that help to iterate over batches and samples for
    # training, validation and testing
    train_batches = DataLoader(train_split, batch_size=train_batch_size, shuffle=True)
    val_batches = DataLoader(val_split, batch_size=train_batch_size, shuffle=True)

    num_train_batches=len(train_batches)
    num_val_batches=len(val_batches)

    train_loss_list = np.zeros((epochs,))
    validation_accuracy_list = np.zeros((epochs,))

    # Define loss function  and optimizer
    # Use Cross Entropy loss from torch.nn to define loss function
    loss_func = nn.CrossEntropyLoss()

    # Use optimizers from torch.optim to define optimizer
    if opt=='SGD':
        optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)
    elif opt=='Adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    elif opt=='RMS':
        optimizer = torch.optim.RMSprop(model.parameters(), lr=learning_rate)

    # Iterate over epochs, batches with progress bar and train+ validate the ACAIGFCN
    # Track the loss and validation accuracy
    for epoch in tqdm.trange(epochs):
        # initialize running loss as 0
        running_loss = 0.0

        # ACAIGFCN Training 
        for train_features, train_labels in train_batches:
            # Set model into training mode
            model.train()
            
            # Reshape images into a vector
            train_features = train_features.reshape(-1, 28*28)

            # Reset gradients, 
            optimizer.zero_grad() 
            #Calculate training loss on model
            outputs = model(train_features)
            loss = loss_func(outputs, train_labels)
            # Perform optimization, back propagation
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
    
        # Record loss for the epoch by dividing sum of running loss for the epoch, dividing by length of train batches
        train_loss_list[epoch] = running_loss / num_train_batches
        
        

        val_acc_sum = 0
        # ACAIGFCN Validation
        for val_features, val_labels in val_batches:
            
            # Telling PyTorch we aren't passing inputs to network for training purpose
            with torch.no_grad(): 
                model.eval()
                
                # Reshape validation images into a vector
                val_features = val_features.reshape(-1, 28*28)
            
                # Compute validation outputs (targets) 
                outputs = model(val_features)  # Forward pass
                _, predicted = torch.max(outputs, 1)  # Get predicted class labels

                # add to lists
                val_acc = accuracy_score(val_labels, predicted)
                val_acc_sum += val_acc
                
                
        # Record accuracy for the epoch; print training loss, validation accuracy
        print("Epoch: "+ str(epoch) +"; Validation Accuracy:" + str(val_acc_sum/num_val_batches*100) + '%')
        validation_accuracy_list[epoch] = val_acc_sum/num_val_batches*100

    return model, train_loss_list, validation_accuracy_list



In [6]:
#Calculate accuracy on test set
def acc_test(model, test_batch_size):
    test_batches = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=True)
    num_test_batches = len(test_batches)

    # Telling PyTorch we aren't passing inputs to network for training purpose
    with torch.no_grad():
        test_acc_sum = 0
        test_acc_arr = np.array([])
        for test_features, test_labels in test_batches:

            model.eval()
            # Reshape test images into a vector
            test_features = test_features.reshape(-1, 28*28)

            # Compute validation outputs (targets) 
            outputs = model(test_features)  # Forward pass
            _, predicted = torch.max(outputs, 1)  # Get predicted class labels
            # and compute accuracy 
            test_acc = accuracy_score(test_labels, predicted)
            test_acc_arr = np.append(test_acc_arr,test_acc)
            test_acc_sum += test_acc
        
        # Compute total (mean) accuracy
        test_acc_mean = (test_acc_sum / num_test_batches) * 100
        test_acc_std = np.std(test_acc_arr)
        # Report total (mean) accuracy, can also compute std based on batches
        print(f'Total mean accuracy is {test_acc_mean} with std of {test_acc_std}')

    return test_acc_mean, test_acc_std

In [12]:
train_batch_size_MNIST = 128
test_batch_size_MNIST = 64
layers_MNIST = [512,256]
lr_MNIST = 0.001
epochs = 50
model_MNIST, MNIST_ex1_train_loss_list, MNIST_ex1_validation_accuracy_list = train_and_eval(hidden_lays=layers_MNIST, learning_rate=lr_MNIST, epochs=epochs, train_batch_size=train_batch_size_MNIST, opt='Adam', dropout_prob=0.1, init='KU', batchnorm='yes')
MNIST_acc_mea, MNIST_acc_std_d = acc_test(model_MNIST, test_batch_size=test_batch_size_MNIST)

  2%|▏         | 1/50 [00:03<02:45,  3.37s/it]

Epoch: 0; Validation Accuracy:96.91536854103344%


  4%|▍         | 2/50 [00:06<02:42,  3.38s/it]

Epoch: 1; Validation Accuracy:97.4354103343465%


  6%|▌         | 3/50 [00:10<02:38,  3.37s/it]

...

 98%|█████████▊| 49/50 [02:58<00:03,  3.59s/it]

Epoch: 48; Validation Accuracy:98.61559650455926%


100%|██████████| 50/50 [03:02<00:00,  3.65s/it]

Epoch: 49; Validation Accuracy:98.57997720364742%


Total mean accuracy is 98.50716560509554 with std of 0.01638706406270125
